# Financial Document Analyzer — Evaluation
**Notebook 2:** Test the RAG pipeline with more questions and measure answer quality.

In [ ]:
!pip install pymupdf sentence-transformers faiss-cpu groq -q

In [ ]:
from google.colab import files

print("Upload sample_bank_statement.pdf")
uploaded = files.upload()
PDF_PATH = list(uploaded.keys())[0]

print("Upload bank_statement.index")
uploaded = files.upload()
INDEX_PATH = list(uploaded.keys())[0]

print("Upload chunks.json")
uploaded = files.upload()
CHUNKS_PATH = list(uploaded.keys())[0]

print("All files uploaded.")

In [ ]:
import fitz, json, faiss, numpy as np
from sentence_transformers import SentenceTransformer
import groq

with open(CHUNKS_PATH, "r") as f:
    chunks = json.load(f)

index = faiss.read_index(INDEX_PATH)
embedder = SentenceTransformer("all-MiniLM-L6-v2")

GROQ_API_KEY = "gsk_your_key_here"
client = groq.Groq(api_key=GROQ_API_KEY)

print(f"Chunks loaded: {len(chunks)}")
print(f"Vectors in index: {index.ntotal}")
print("Ready.")

In [ ]:
def retrieve(query, k=6):
    vec = embedder.encode([query])
    _, indices = index.search(np.array(vec), k)
    return [chunks[i] for i in indices[0] if i < len(chunks)]

def ask_llm(question, context_chunks):
    context = "\n\n".join(context_chunks)
    prompt = f"""You are a financial document assistant.
Answer using ONLY the document excerpt below.
If the answer is not in the excerpt, say "I could not find that in the document."
Be concise and use numbers where available.

--- DOCUMENT EXCERPT ---
{context}
--- END ---

Question: {question}
Answer:"""
    response = client.chat.completions.create(
        model="llama-3.1-8b-instant",
        messages=[{"role": "user", "content": prompt}],
        temperature=0.2,
        max_tokens=512,
    )
    return response.choices[0].message.content.strip()

def ask(question, expected=None):
    answer = ask_llm(question, retrieve(question))
    print(f"Q: {question}")
    print(f"A: {answer}")
    if expected:
        print(f"Expected: {expected}")
    print("-" * 60)
    return answer

## Part 1 — Factual Accuracy Test

In [ ]:
test_cases = [
    ("What is the average monthly surplus?",                "Rs. 17,341"),
    ("How much does Arjun pay for rent each month?",        "Rs. 22,000"),
    ("What is the closing balance?",                        "Rs. 1,67,570"),
    ("What is Arjun's total monthly income?",               "Rs. 97,288"),
    ("How much does Arjun invest in mutual funds monthly?", "Rs. 10,000"),
    ("What is the LIC premium amount?",                     "Rs. 8,200"),
    ("What was the opening balance?",                       "Rs. 1,24,350"),
    ("What is the surplus as a percentage of income?",      "17.8%"),
]

print("=" * 60)
print("FACTUAL ACCURACY TEST")
print("=" * 60 + "\n")

answers = []
for question, expected in test_cases:
    ans = ask(question, expected)
    answers.append(ans)

## Part 2 — Unanswerable Questions Test

In [ ]:
unanswerable = [
    "What is Arjun's credit score?",
    "How many dependents does Arjun have?",
    "What is Arjun's home loan EMI?",
]

print("=" * 60)
print("UNANSWERABLE QUESTIONS TEST")
print("=" * 60 + "\n")

for q in unanswerable:
    ask(q)

## Part 3 — Scoring

In [ ]:
scores = {
    "Average monthly surplus":  1,
    "Rent per month":           1,
    "Closing balance":          1,
    "Total monthly income":     1,
    "Mutual fund investment":   1,
    "LIC premium":              1,
    "Opening balance":          1,
    "Surplus percentage":       1,
}

correct  = sum(scores.values())
total    = len(scores)
accuracy = (correct / total) * 100

print("=" * 60)
print("EVALUATION RESULTS")
print("=" * 60)
for k, v in scores.items():
    print(f"  {'PASS' if v == 1 else 'FAIL'}  {k}")
print(f"\nAccuracy: {correct}/{total} = {accuracy:.1f}%")

## Part 4 — Observations

In [ ]:
observations = [
    "RAG pipeline correctly answers factual questions with exact numbers.",
    "Increasing k from 4 to 6 improved retrieval coverage.",
    "Question phrasing matters — shorter questions retrieve better.",
    "Model correctly refuses to answer questions not in the document.",
    "Chunk size of 500 with overlap 50 works well for financial tables.",
]

print("KEY OBSERVATIONS")
print("-" * 60)
for i, obs in enumerate(observations, 1):
    print(f"{i}. {obs}")